# sparse-cl trên Kaggle

**Settings cần bật**: Accelerator = GPU, Internet = ON (để `timm` tải trọng số ViT).

Upload thư mục `sparse-cl/` thành một Kaggle Dataset rồi Add vào notebook này.

Ba cấu hình được chạy:
1. `(backbone → chiếu thưa)` đóng băng → MLP(ReLU) — chỉ học MLP
2. backbone đóng băng → chiếu thưa → MLP(ReLU) — học **chiếu + MLP**
3. backbone đóng băng → chiếu thưa → cls tuyến tính — học **chiếu**

In [ ]:
import os, glob, shutil, subprocess, sys, torch

src = os.path.dirname(sorted(glob.glob('/kaggle/input/**/train.py', recursive=True), key=len)[0])
os.chdir('/kaggle/working'); shutil.rmtree('sparse-cl', ignore_errors=True)
shutil.copytree(src, 'sparse-cl'); os.chdir('sparse-cl')

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.16'], check=True)
    import timm

print('torch :', torch.__version__, '| timm:', timm.__version__)
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Bật GPU: Settings > Accelerator > GPU'

## 1. Trích feature một lần rồi cache

Feature **không phụ thuộc seed** — chỉ cách chia task mới phụ thuộc. Nên cache theo dataset, dùng lại cho mọi cấu hình và mọi seed.

Commit output cell này thành Dataset để các notebook sau khỏi phải chạy ViT nữa (khi đó chạy được cả trên session CPU, không tốn quota GPU).

In [ ]:
import torch
from config import get_parser, validate
from data import cached_features
from train import load_backbone, cache_exists

args = validate(get_parser().parse_args([
    '--dataset', 'CIFAR-100', '--root', '/kaggle/working/data',
    '--num_classes', '100', '--num_tasks', '10',
    '--data_augmentation', 'vit', '--cache_dir', '/kaggle/working/cache',
]))
device = torch.device('cuda:0')

bb = None if cache_exists(args) else load_backbone(args.model_name, device)
Xtr, Ytr, Xte, Yte = cached_features(args, bb, device)
print('train', tuple(Xtr.shape), '| test', tuple(Xte.shape))
del bb; torch.cuda.empty_cache()

## 2. Chạy ba cấu hình

Sau khi có cache, mỗi cấu hình chỉ mất vài phút.

In [ ]:
COMMON = ('--dataset CIFAR-100 --root /kaggle/working/data --cache_dir /kaggle/working/cache '
          '--out_dir /kaggle/working/runs --num_classes 100 --num_tasks 10 '
          '--data_augmentation vit --expand_dim 10000 --synaptic_degree 300 '
          '--coding_level 0.1 --epochs 100 --batch_size 256 --seed 1993 --gpu 0')

# GIAI DOAN 1 - chua dung EWC (--cl_reg none cho tat ca).
# Luoi 2x2:              khong MLP     co MLP(ReLU)
#   chieu DONG BANG          0              1
#   chieu HOC DUOC           3              2
FROZEN = '--train_projection False --projection_schedule task0'
LEARN  = '--train_projection True  --projection_schedule continual'
MLP    = '--use_mlp True --mlp_act relu --mlp_hidden 512'
NOMLP  = '--use_mlp False'

CONFIGS = {
    '0_frozen_linear': f'--cl_reg none {FROZEN} {NOMLP}',
    '1_frozen_mlp':    f'--cl_reg none {FROZEN} {MLP}',
    '2_learn_mlp':     f'--cl_reg none {LEARN} {MLP}',
    '3_learn_linear':  f'--cl_reg none {LEARN} {NOMLP}',
}

for name, flags in CONFIGS.items():
    print('\n' + '#' * 70 + f'\n### {name}\n' + '#' * 70)
    !python train.py {COMMON} {flags}

# GIAI DOAN 2 (chay sau, khi da co so lieu giai doan 1): bat EWC-DR tren cau
# hinh tot nhat de do phan tang them. Theo doi pen_over_clf de chinh lamda.
#   !python train.py {COMMON} {LEARN} {NOMLP} --cl_reg ewc_dr --lamda 10000

## 3. Đọc kết quả

Mốc so sánh: Fly-CL trên cùng máy, cùng seed 1993 → **A_T = 88.68**, **Ā = 92.99**
(từ `Fly-CL-main/log_cifar_seed1993.txt`).

Giai đoạn này chưa bật EWC nên `pen/clf` và `omega_sat` sẽ trống — đúng như mong đợi.
Hai cột cần nhìn cùng với accuracy:

- `forget` — chỉ số quan trọng nhất ở giai đoạn này. Cấu hình nào giữ được task cũ tốt hơn?
- `dead_frac` — tỉ lệ unit không bao giờ thắng top-k. Tăng dần thì bật `--adaptive_threshold`.
  Accuracy **không** báo cho biết điều này.

Đọc theo cặp:
- **1 ↔ 3** — học phép chiếu mang lại gì (cả hai đều có/không MLP khác nhau, xem lưu ý dưới)
- **2 ↔ 3** — MLP mang lại gì, khi phép chiếu đều được học

Lưu ý: cấu hình 1 và 3 khác nhau **hai** thứ (chiếu đóng băng/học được, và có/không MLP).
Muốn tách bạch hoàn toàn thì chạy thêm cấu hình thứ tư — chiếu đóng băng + cls tuyến tính:

```
--train_projection False --projection_schedule task0 --use_mlp False --cl_reg none
```

In [ ]:
import json, glob

rows = []
for f in sorted(glob.glob('/kaggle/working/runs/*.json')):
    d = json.load(open(f)); m = d['metrics']; last = d['per_task'][-1]
    rows.append((f.split('/')[-1][:52], m['A_T'], m['A_bar'], m['forgetting'],
                 last.get('pen_over_clf', '-'), last.get('omega_saturated', '-'),
                 last.get('dead_frac', '-')))

hdr = ('run', 'A_T', 'A_bar', 'forget', 'pen/clf', 'omega_sat', 'dead')
print(f'{hdr[0]:<52}' + ''.join(f'{h:>11}' for h in hdr[1:]))
print('-' * 118)
for r in rows:
    print(f'{r[0]:<52}' + ''.join(f'{str(v):>11}' for v in r[1:]))
print('-' * 118)
print(f'{"Fly-CL (moc so sanh, seed 1993)":<52}{88.68:>11}{92.99:>11}')